# Pipeline 04Ga: global whole-model template baseline (deterministic)

Deterministic, non-LLM whole-model summary per model (`[MODEL]` / `[DRIVERS]` / `[RECOMMENDATION]`) built from the global importance +
shape-curve JSONs. Global analogue of the local per-instance template baseline (`04La`). No API call, $0.

Output: `results/04Ga/global_{model}.json`. Split out of `04La` (02.07.).

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import RESULTS_DIR, EXPLANATIONS_DIR

LOSS_KEY   = 'poisson_log'
XAI_MODELS = ['xgb', 'ebm']

OUT_DIR = RESULTS_DIR / '04Ga'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Models: {XAI_MODELS}')
print(f'Output: {OUT_DIR}')

In [ ]:
# Same mappings as build_judge_prompt in NB 05.
WEEKDAYS = {
    0: 'Sunday', 1: 'Monday', 2: 'Tuesday', 3: 'Wednesday',
    4: 'Thursday', 5: 'Friday', 6: 'Saturday',
}
MONTHS = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April',
    5: 'May', 6: 'June', 7: 'July', 8: 'August',
    9: 'September', 10: 'October', 11: 'November', 12: 'December',
}
WEATHER = {
    1: 'clear/few clouds',
    2: 'mist/cloudy',
    3: 'light rain/snow',
    4: 'heavy rain/thunderstorm',
}
FEATURE_NAMES = {
    'hr':         'hour',
    'temp':       'temperature',
    'hum':        'humidity',
    'windspeed':  'wind speed',
    'yr':         'year',
    'mnth':       'month',
    'weekday':    'weekday',
    'weathersit': 'weather',
    'holiday':    'holiday status',
}


def readable_value(feature: str, raw: float) -> str:
    """Denormalise a raw feature value into a readable string."""
    if feature == 'hr':
        return f'{int(raw):02d}:00'
    elif feature == 'temp':
        return f'~{raw * 41:.1f} C'
    elif feature == 'hum':
        return f'{raw * 100:.0f} %'
    elif feature == 'windspeed':
        return f'{raw * 67:.1f} km/h'
    elif feature == 'yr':
        return '2011' if int(raw) == 0 else '2012'
    elif feature == 'mnth':
        return MONTHS.get(int(raw), str(int(raw)))
    elif feature == 'weekday':
        return WEEKDAYS.get(int(raw), str(int(raw)))
    elif feature == 'weathersit':
        return WEATHER.get(int(raw), str(int(raw)))
    elif feature == 'holiday':
        return 'holiday' if int(raw) == 1 else 'no holiday'
    else:
        return str(raw)


print('Helper tables loaded.')

In [ ]:
# --- Global whole-model baseline: [MODEL] / [DRIVERS] / [RECOMMENDATION] ---
# The local [PREDICTION] section is repurposed into a global [MODEL] overview
# (design decision, see planning/Revision_30062026_Umsetzungsplan.md). Deterministic.
import math

# Discrete-coded features: readable_value() truncates via int(), so continuous shape
# curves (bin midpoints like 0.75) must be rounded to hit the right category label.
_DISCRETE_FEATURES = {'hr', 'yr', 'mnth', 'weekday', 'weathersit', 'holiday'}


def _load_curve(xai: str, feature: str) -> dict:
    return json.loads((EXPLANATIONS_DIR / f'global_curve_{xai}_{feature}.json').read_text())


def _curve_direction(curve: dict) -> str:
    """Coarse shape of a global term curve: rising / falling / mixed / flat."""
    y = curve['y']
    if len(y) < 2:
        return 'flat'
    peak_i   = max(range(len(y)), key=lambda i: y[i])
    trough_i = min(range(len(y)), key=lambda i: y[i])
    # An extreme in the interior means the effect is non-monotonic.
    if 0 < peak_i < len(y) - 1 or 0 < trough_i < len(y) - 1:
        return 'mixed'
    return 'rising' if y[-1] >= y[0] else 'falling'


def _peak_label(curve: dict) -> str:
    """Readable feature value at which the contribution is highest."""
    y, x, feat = curve['y'], curve['x'], curve['feature']
    raw = float(x[max(range(len(y)), key=lambda k: y[k])])
    if feat in _DISCRETE_FEATURES:
        raw = round(raw)
    return readable_value(feat, raw)


def generate_global_template(global_data: dict, xai: str) -> str:
    """Deterministic whole-model description of an EBM/XGB from the global JSON.
    Structure: [MODEL] / [DRIVERS] / [RECOMMENDATION] (global analogue of the local
    [PREDICTION] / [DRIVERS] / [RECOMMENDATION] template)."""
    task     = global_data['task']['description']
    baseline = math.exp(global_data['base_value'])   # log space -> cnt space
    metrics  = global_data.get('metrics', {})
    top3     = global_data['global_importance'][:3]

    # [MODEL] section (repurposed from the local [PREDICTION] section)
    fit_keys = ('rmse', 'mae', 'r2', 'poisson_deviance')
    metric_str = ', '.join(f'{k}={metrics[k]:.3f}' for k in fit_keys if k in metrics) or 'n/a'
    model = (
        f'[MODEL] The model predicts {task[0].lower() + task[1:]}. On average it expects '
        f'about {baseline:.0f} rentals per hour (baseline). Model fit: {metric_str}.'
    )

    # [DRIVERS] section: top-3 features by global importance + shape from the curves
    lines = []
    for item in top3:
        feat  = item['feature']
        curve = _load_curve(xai, feat)
        lines.append(
            f"* {FEATURE_NAMES.get(feat, feat)} (rank {item['rank']}, "
            f"importance {item['importance']:.3f}): {_curve_direction(curve)} effect, "
            f'contribution highest around {_peak_label(curve)}.'
        )
    drivers = (
        '[DRIVERS] The features with the largest overall influence on demand are:\n'
        + '\n'.join(lines)
    )

    # [RECOMMENDATION] section: general, tied to the dominant driver
    top      = top3[0]
    top_name = FEATURE_NAMES.get(top['feature'], top['feature'])
    top_peak = _peak_label(_load_curve(xai, top['feature']))
    recommendation = (
        f'[RECOMMENDATION] Plan capacity around the dominant driver ({top_name}): '
        f'provide the most bikes and staff at its high-demand level (around {top_peak}) '
        f'and scale resources down otherwise.'
    )

    return '\n\n'.join([model, drivers, recommendation])


# Smoke test
_gdata = json.loads((EXPLANATIONS_DIR / f'global_ebm_{LOSS_KEY}.json').read_text())
print(generate_global_template(_gdata, 'ebm'))

In [ ]:
# Run the global baseline for both models. Deterministic, no API, $0.
for xai in XAI_MODELS:
    gdata = json.loads((EXPLANATIONS_DIR / f'global_{xai}_{LOSS_KEY}.json').read_text())

    t0          = time.perf_counter()
    explanation = generate_global_template(gdata, xai)
    elapsed     = round(time.perf_counter() - t0, 6)

    record = {
        'explanation':  explanation,
        'elapsed_s':    elapsed,
        'usage':        {'input_tokens': 0, 'output_tokens': 0, 'cache_read_input_tokens': 0},
        'n_tool_calls': 0,
        'scope':        'global',
        'model':        xai,
    }
    out_file = OUT_DIR / f'global_{xai}.json'
    out_file.write_text(json.dumps(record, indent=2, ensure_ascii=False))
    print(f'  {xai.upper()} global: {len(explanation.split())} words -> {out_file.name}')

print(f'\nDone. Global baselines saved in {OUT_DIR} (no API call, $0).')